In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Vytvoření pluginu pro Transpiler

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vytvořen s použitím následujících požadavků.
Doporučujeme používat tyto verze nebo novější.

```
qiskit[all]~=2.3.0
```
</details>
Vytvoření [pluginu pro Transpiler](transpiler-plugins) je skvělý způsob, jak sdílet svůj transpilační kód s širší komunitou Qiskit a umožnit ostatním uživatelům těžit z funkcionalit, které jsi vyvinul/a. Děkujeme za tvůj zájem přispívat do komunity Qiskit!

Před vytvořením pluginu pro Transpiler se musíš rozhodnout, jaký druh pluginu je pro tvou situaci vhodný. Existují tři druhy pluginů pro Transpiler:
- [**Plugin pro fázi Transpileru**](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins). Zvol tuto možnost, pokud definuješ pass manager, který lze nahradit za jednu z [6 fází](transpiler-stages) nastaveného stagovaného pass manageru.
- [**Plugin pro syntézu unitárních operátorů**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin). Zvol tuto možnost, pokud tvůj transpilační kód přijímá jako vstup unitární matici (reprezentovanou jako pole Numpy) a vrací popis kvantového Circuit implementujícího tuto unitární matici.
- [**Plugin pro vysokoúrovňovou syntézu**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin). Zvol tuto možnost, pokud tvůj transpilační kód přijímá jako vstup „vysokoúrovňový objekt", například operátor Clifford nebo lineární funkci, a vrací popis kvantového Circuit implementujícího tento vysokoúrovňový objekt. Vysokoúrovňové objekty jsou reprezentovány podtřídami třídy [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation).

Jakmile určíš, jaký druh pluginu chceš vytvořit, postupuj podle těchto kroků:

1. Vytvoř podtřídu příslušné abstraktní třídy pluginu:
   - [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin) pro plugin fáze Transpileru,
   - [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) pro plugin syntézy unitárních operátorů a
   - [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) pro plugin vysokoúrovňové syntézy.
2. Vystav třídu jako [vstupní bod setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) v metadatech balíčku, obvykle úpravou souboru `pyproject.toml`, `setup.cfg` nebo `setup.py` svého Python balíčku.

Neexistuje žádné omezení počtu pluginů, které může jeden balíček definovat, ale každý plugin musí mít jedinečný název. Samotné Qiskit SDK obsahuje řadu pluginů, jejichž názvy jsou také rezervovány. Rezervované názvy jsou:

- Pluginy fází Transpileru: Viz [tato tabulka](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#plugin-stages).
- Pluginy syntézy unitárních operátorů: `default`, `aqc`, `sk`
- Pluginy vysokoúrovňové syntézy:

| Třída operace | Název operace | Rezervované názvy |
| --- | --- | --- |
| [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford#clifford) | `clifford` | `default`, `ag`, `bm`, `greedy`, `layers`, `lnn` |
| [LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction#linearfunction) | `linear_function` | `default`, `kms`, `pmh` |
| [PermutationGate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PermutationGate#permutationgate) | `permutation` | `default`, `kms`, `basic`, `acg`, `token_swapper` |

V následujících sekcích ukážeme příklady těchto kroků pro různé typy pluginů. V těchto příkladech předpokládáme, že vytváříme Python balíček s názvem `my_qiskit_plugin`. Informace o vytváření Python balíčků najdeš v [tomto tutoriálu](https://packaging.python.org/en/latest/tutorials/packaging-projects/) na webu Pythonu.
## Příklad: Vytvoření pluginu fáze Transpileru
V tomto příkladu vytvoříme plugin fáze Transpileru pro fázi `layout` (popis 6 fází vestavěného transpilačního pipeline Qiskitu najdeš v části [Fáze Transpileru](transpiler-stages)).
Náš plugin jednoduše spouští [VF2Layout](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.VF2Layout) s počtem pokusů závislým na požadované úrovni optimalizace.

Nejprve vytvoříme podtřídu [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin). Potřebujeme implementovat jednu metodu s názvem [`pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin#pass_manager). Tato metoda přijímá jako vstup [PassManagerConfig](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManagerConfig) a vrací pass manager, který definujeme. Objekt PassManagerConfig uchovává informace o cílovém Backend, například jeho coupling map a základní Gate.

In [1]:
# This import is needed for python versions prior to 3.10
from __future__ import annotations

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import VF2Layout
from qiskit.transpiler.passmanager_config import PassManagerConfig
from qiskit.transpiler.preset_passmanagers import common
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePlugin,
)


class MyLayoutPlugin(PassManagerStagePlugin):
    def pass_manager(
        self,
        pass_manager_config: PassManagerConfig,
        optimization_level: int | None = None,
    ) -> PassManager:
        layout_pm = PassManager(
            [
                VF2Layout(
                    coupling_map=pass_manager_config.coupling_map,
                    properties=pass_manager_config.backend_properties,
                    max_trials=optimization_level * 10 + 1,
                    target=pass_manager_config.target,
                )
            ]
        )
        layout_pm += common.generate_embed_passmanager(
            pass_manager_config.coupling_map
        )
        return layout_pm

Nyní vystavíme plugin přidáním vstupního bodu do metadat našeho Python balíčku.
Předpokládáme, že třída, kterou jsme definovali, je vystavena v modulu s názvem `my_qiskit_plugin`, například importem v souboru `__init__.py` modulu `my_qiskit_plugin`.
Upravíme soubor `pyproject.toml`, `setup.cfg` nebo `setup.py` našeho balíčku (podle toho, jaký soubor jsi zvolil/a pro uložení metadat svého Python projektu):

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.transpiler.layout"]
    "my_layout" = "my_qiskit_plugin:MyLayoutPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.transpiler.layout =
        my_layout = my_qiskit_plugin:MyLayoutPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.transpiler.layout': [
                'my_layout = my_qiskit_plugin:MyLayoutPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>
Viz [tabulka fází pluginů pro Transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#stage-table), kde najdeš vstupní body a požadavky pro každou fázi Transpileru.

Chceš-li ověřit, že Qiskit tvůj plugin úspěšně detekuje, nainstaluj balíček pluginu a postupuj podle pokynů v části [Pluginy pro Transpiler](transpiler-plugins#list-available-transpiler-stage-plugins) pro výpis nainstalovaných pluginů, a ujisti se, že tvůj plugin se v seznamu zobrazí:

In [2]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

If our example plugin were installed, then the name `my_layout` would appear in this list.

If you want to use a built-in transpiler stage as the starting point for your transpiler stage plugin, you can obtain the pass manager for a built-in transpiler stage using [PassManagerStagePluginManager](/docs/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). The following code cell shows how to do this to obtain the built-in optimization stage for optimization level 3.

In [3]:
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePluginManager,
)

# Initialize the plugin manager
plugin_manager = PassManagerStagePluginManager()

# Here we create a pass manager config to use as an example.
# Instead, you should use the pass manager config that you already received as input
# to the pass_manager method of your PassManagerStagePlugin.
pass_manager_config = PassManagerConfig()

# Obtain the desired built-in transpiler stage
optimization = plugin_manager.get_passmanager_stage(
    "optimization", "default", pass_manager_config, optimization_level=3
)

Pokud by byl náš ukázkový plugin nainstalován, v tomto seznamu by se zobrazil název `my_layout`.

Chceš-li jako výchozí bod pro svůj plugin fáze Transpileru použít vestavěnou fázi Transpileru, můžeš získat pass manager pro vestavěnou fázi Transpileru pomocí [PassManagerStagePluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). Následující buňka kódu ukazuje, jak to provést pro získání vestavěné fáze optimalizace pro úroveň optimalizace 3.

In [4]:
import numpy as np
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit.dagcircuit import DAGCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler.passes import UnitarySynthesis
from qiskit.transpiler.passes.synthesis.plugin import UnitarySynthesisPlugin


class MyUnitarySynthesisPlugin(UnitarySynthesisPlugin):
    @property
    def supports_basis_gates(self):
        # Returns True if the plugin can target a list of basis gates
        return True

    @property
    def supports_coupling_map(self):
        # Returns True if the plugin can synthesize for a given coupling map
        return False

    @property
    def supports_natural_direction(self):
        # Returns True if the plugin supports a toggle for considering
        # directionality of 2-qubit gates
        return False

    @property
    def supports_pulse_optimize(self):
        # Returns True if the plugin can optimize pulses during synthesis
        return False

    @property
    def supports_gate_lengths(self):
        # Returns True if the plugin can accept information about gate lengths
        return False

    @property
    def supports_gate_errors(self):
        # Returns True if the plugin can accept information about gate errors
        return False

    @property
    def supports_gate_lengths_by_qubit(self):
        # Returns True if the plugin can accept information about gate lengths
        # (The format of the input differs from supports_gate_lengths)
        return False

    @property
    def supports_gate_errors_by_qubit(self):
        # Returns True if the plugin can accept information about gate errors
        # (The format of the input differs from supports_gate_errors)
        return False

    @property
    def min_qubits(self):
        # Returns the minimum number of qubits the plugin supports
        return None

    @property
    def max_qubits(self):
        # Returns the maximum number of qubits the plugin supports
        return None

    @property
    def supported_bases(self):
        # Returns a dictionary of supported bases for synthesis
        return None

    def run(self, unitary: np.ndarray, **options) -> DAGCircuit:
        basis_gates = options["basis_gates"]
        synth_pass = UnitarySynthesis(basis_gates, min_qubits=3)
        qubits = QuantumRegister(3)
        circuit = QuantumCircuit(qubits)
        circuit.append(Operator(unitary).to_instruction(), qubits)
        dag_circuit = synth_pass.run(circuit_to_dag(circuit))
        return dag_circuit

## Příklad: Vytvoření pluginu pro syntézu unitárních operátorů
V tomto příkladu vytvoříme plugin pro syntézu unitárních operátorů, který jednoduše používá vestavěný transpilační průchod [UnitarySynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.UnitarySynthesis#unitarysynthesis) pro syntézu Gate. Tvůj vlastní plugin samozřejmě bude dělat něco zajímavějšího.

Třída [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) definuje rozhraní a kontrakt pro pluginy syntézy unitárních operátorů. Primární metodou je
[`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run),
která přijímá jako vstup pole Numpy uchovávající unitární matici
a vrací [DAGCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.dagcircuit.DAGCircuit) reprezentující Circuit syntetizovaný z této unitární matice.
Kromě metody `run` je třeba definovat řadu property metod.
Dokumentaci všech požadovaných vlastností najdeš v [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin).

Pojďme vytvořit naši podtřídu UnitarySynthesisPlugin:

In [5]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

Pokud zjistíš, že vstupy dostupné metodě [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run)
pro tvé účely nestačí, prosím [otevři issue](https://github.com/Qiskit/qiskit/issues/new/choose) s popisem svých požadavků. Změny rozhraní pluginu, jako je přidání dalších volitelných vstupů, budou provedeny zpětně kompatibilním způsobem, takže nebudou vyžadovat změny v existujících pluginech.

> **Note:** Všechny metody s předponou `supports_` jsou na odvozené třídě `UnitarySynthesisPlugin` vyhrazeny jako součást rozhraní. Na podtřídě bys neměl/a definovat žádné vlastní metody `supports_*`, které nejsou definovány v abstraktní třídě.

Nyní vystavíme plugin přidáním vstupního bodu do metadat našeho Python balíčku.
Předpokládáme, že třída, kterou jsme definovali, je vystavena v modulu s názvem `my_qiskit_plugin`, například importem v souboru `__init__.py` modulu `my_qiskit_plugin`.
Upravíme soubor `pyproject.toml`, `setup.cfg` nebo `setup.py` našeho balíčku:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.unitary_synthesis"]
    "my_unitary_synthesis" = "my_qiskit_plugin:MyUnitarySynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.unitary_synthesis =
        my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.unitary_synthesis': [
                'my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

Stejně jako dříve, pokud tvůj projekt používá `setup.cfg` nebo `setup.py` místo `pyproject.toml`, viz [dokumentace setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html), kde najdeš, jak tyto řádky upravit pro svou situaci.

Chceš-li ověřit, že Qiskit tvůj plugin úspěšně detekuje, nainstaluj balíček pluginu a postupuj podle pokynů v části [Pluginy pro Transpiler](transpiler-plugins#list-available-unitary-synthesis-plugins) pro výpis nainstalovaných pluginů, a ujisti se, že tvůj plugin se v seznamu zobrazí:

In [6]:
from qiskit.synthesis import synth_clifford_bm
from qiskit.transpiler.passes.synthesis.plugin import HighLevelSynthesisPlugin


class MyCliffordSynthesisPlugin(HighLevelSynthesisPlugin):
    def run(
        self,
        high_level_object,
        coupling_map=None,
        target=None,
        qubits=None,
        **options,
    ) -> QuantumCircuit:
        if high_level_object.num_qubits <= 3:
            return synth_clifford_bm(high_level_object)
        else:
            return None

This plugin synthesizes objects of type [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) that have
at most 3 qubits, using the `synth_clifford_bm` method.

Now, we expose the plugin by adding an entry point in our Python package metadata.
Here, we assume that the class we defined is exposed in a module called `my_qiskit_plugin`, for example by being imported in the `__init__.py` file of the `my_qiskit_plugin` module.
We edit the `pyproject.toml`, `setup.cfg`, or `setup.py` file of our package:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.synthesis"]
    "clifford.my_clifford_synthesis" = "my_qiskit_plugin:MyCliffordSynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.synthesis =
        clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.synthesis': [
                'clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

The `name` consists of two parts separated by a dot (`.`):
- The name of the type of [Operation](/docs/api/qiskit/qiskit.circuit.Operation) that the plugin synthesizes (in this case, `clifford`). Note that this string corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the Operation class, and not the name of the class itself.
- The name of the plugin (in this case, `special`).

As before, if your project uses `setup.cfg` or `setup.py` instead of `pyproject.toml`, see the [setuptools documentation](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) for how to adapt these lines for your situation.

To check that your plugin is successfully detected by Qiskit, install your plugin package and follow the instructions at [Transpiler plugins](transpiler-plugins#list-available-high-level-synthesis-plugins) for listing installed plugins, and ensure that your plugin appears in the list:

In [7]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

Pokud by byl náš ukázkový plugin nainstalován, v tomto seznamu by se zobrazil název `my_unitary_synthesis`.

Aby se vyhovělo pluginům pro syntézu unitárních operátorů, které nabízejí více možností,
rozhraní pluginu obsahuje volbu, kde mohou uživatelé poskytnout slovník s volnou konfigurací. Ten bude předán metodě `run`
prostřednictvím klíčového argumentu `options`. Pokud tvůj plugin tyto konfigurační možnosti má, měl/a bys je jasně zdokumentovat.

## Příklad: Vytvoření pluginu pro vysokoúrovňovou syntézu

V tomto příkladu vytvoříme plugin pro vysokoúrovňovou syntézu, který jednoduše používá vestavěnou funkci [synth_clifford_bm](https://docs.quantum.ibm.com/api/qiskit/synthesis#synth_clifford_bm) pro syntézu operátoru Clifford.

Třída [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) definuje rozhraní a kontrakt pro pluginy vysokoúrovňové syntézy. Primární metodou je [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin#run).
Poziční argument `high_level_object` je [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) reprezentující „vysokoúrovňový" objekt, který má být syntetizován. Může to být například
[LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) nebo
[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford).
Jsou přítomny následující klíčové argumenty:
- `target` specifikuje cílový Backend a umožňuje pluginu
přistupovat ke všem informacím specifickým pro cíl,
jako je coupling map, podporovaná sada Gate a podobně
- `coupling_map` specifikuje pouze coupling map a používá se pouze tehdy, když `target` není zadán.
- `qubits` specifikuje seznam Qubitů, nad kterými je
vysokoúrovňový objekt definován, v případě, že syntéza probíhá na fyzickém Circuit.
Hodnota ``None`` znamená, že rozložení ještě nebylo zvoleno a fyzické Qubity v cíli nebo coupling mapě, na nichž tato operace pracuje, ještě nebyly určeny.
- `options`, slovník s volnou konfigurací pro volby specifické pro plugin. Pokud tvůj plugin tyto konfigurační možnosti má,
měl/a bys je jasně zdokumentovat.

Metoda `run` vrací [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)
reprezentující Circuit syntetizovaný z daného vysokoúrovňového objektu.
Je také povoleno vrátit `None`, což znamená, že plugin není schopen syntetizovat daný vysokoúrovňový objekt.
Samotná syntéza vysokoúrovňových objektů je prováděna průchodem Transpileru
[HighLevelSynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.HighLevelSynthesis).

Kromě metody `run` je třeba definovat řadu property metod.
Dokumentaci všech požadovaných vlastností najdeš v [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin).

Pojďme definovat naši podtřídu HighLevelSynthesisPlugin: